In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [ ]:
# Set the working directory + add eomt/ to sys.path
# eval/ holds this notebook and the utilities; eomt/ holds the EoMT package code.
import os, sys

EVAL_DIR = '/content/drive/My Drive/Comprehensive Road Scene Understanding for Autonomous Driving/MaskArchitectureAnomaly_CourseProject/eval'
EOMT_DIR = '/content/drive/My Drive/Comprehensive Road Scene Understanding for Autonomous Driving/MaskArchitectureAnomaly_CourseProject/eomt'

if EOMT_DIR not in sys.path:
    sys.path.insert(0, EOMT_DIR)

assert os.path.isdir(EVAL_DIR), f"Directory doesn't exist": {EVAL_DIR}'
os.chdir(EVAL_DIR)
if EVAL_DIR not in sys.path:
    sys.path.insert(0, EVAL_DIR)

print('CWD:', os.getcwd())
print('erfnet.py in directory:', os.path.exists('erfnet.py'))
print('eomt in path:', EOMT_DIR in sys.path)


In [ ]:
# Dependencies
# Recent peft (>= ~0.15) needs torchao >= 0.16, but Colab's torch is too old for it.
# Fix: pin peft to a version that comes before that check (< 0.15).
import subprocess, sys as _sys

# Base packages
subprocess.run(
    [_sys.executable, '-m', 'pip', 'install', '-q',
     'pyyaml', 'lightning', 'torchmetrics', 'huggingface_hub', 'torchvision'],
    check=True,
)

# Force a peft version that works with Colab's torchao.
# --force-reinstall in case an earlier cell already pulled in peft 0.15+.
subprocess.run(
    [_sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps',
     'peft<0.15'],
    check=True,
)

# Just in case peft >= 0.15 is still around, skip its torchao check before importing.
import importlib, importlib.metadata as md
try:
    import peft.import_utils as _piu
    if hasattr(_piu, 'is_torchao_available'):
        _piu.is_torchao_available = lambda: False
except Exception as _e:
    print('peft.import_utils patch skipped:', _e)

# Check versions
for pkg in ('torchao', 'peft', 'torch'):
    try:
        print(f'{pkg}: {md.version(pkg)}')
    except md.PackageNotFoundError:
        print(f'{pkg}: NOT INSTALLED')

# Quick test of the import that used to fail
from peft import LoraConfig, get_peft_model  # noqa
print('peft import OK')


In [ ]:
# Imports, seed, fpr_at_95_tpr
import gc, glob, random
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import average_precision_score, roc_curve
from torch.amp.autocast_mode import autocast

# Seed just for a reproducible sub-sampling order; cudnn stays fast
# (non-deterministic) since here we only evaluate, we don't train.
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


def fpr_at_95_tpr(scores, labels):
    '''FPR when the TPR reaches 95%. Standard OoD metric: lower = better.'''
    fpr, tpr, _ = roc_curve(labels, scores)
    # searchsorted finds the first index in tpr where tpr >= 0.95
    idx = min(np.searchsorted(tpr, 0.95, side='left'), len(fpr) - 1)
    return float(fpr[idx])


In [ ]:
# Model paths + loader helper with auto-detect (incl. LoRA)
import warnings
from models.eomt import EoMT
from models.vit import ViT
from training.mask_classification_semantic import MaskClassificationSemantic

PROJECT_ROOT = '/content/drive/My Drive/Comprehensive Road Scene Understanding for Autonomous Driving'
MODELS_DIR   = f'{PROJECT_ROOT}/MaskArchitectureAnomaly_CourseProject/trained_models'

CS_CKPT   = f'{MODELS_DIR}/eomt_cityscapes.bin'
COCO_CKPT = f'{MODELS_DIR}/eomt_coco.bin'
# From Step 5: the LoRA experiment (exp_lora) gave the best Cityscapes results
# in Step 4 , so this is the fine-tuned model we evaluate here.
FT_CKPT   = f'{EOMT_DIR}/logs/exp_lora/version_0/checkpoints/epoch=epoch=7-step=step=23800.ckpt'

# Tuple: (name, checkpoint_path, native_num_classes, lora_flag).
# native_num_classes = how many classes the model knows in its original space
# (19 for Cityscapes, 133 for COCO panoptic). The COCO->CS remap cell uses this
# to decide whether to marginalize 133->19 before scoring.
CHECKPOINTS = [
    ('EoMT-Cityscapes',     CS_CKPT,   19,  False),
    ('EoMT-COCO',           COCO_CKPT, 133, False),
    ('EoMT-FineTuned-LoRA', FT_CKPT,   19,  True),
]

for name, path, _, _ in CHECKPOINTS:
    print('OK     ' if os.path.exists(path) else 'MISSING', '-', name)


def _detect_img_size_and_num_q(state_dict, fallback_img=(640, 640), fallback_num_q=200):
    '''Find pos_embed and q.weight in the state_dict, handling the PEFT prefix
    (`base_model.model.`) that the LoRA wrapping adds to the encoder.
    Returns (img_size, num_queries) matching the checkpoint.'''
    pe_key = next((k for k in state_dict if k.endswith('backbone.pos_embed')), None)
    q_key  = next((k for k in state_dict if k.endswith('network.q.weight') or k == 'network.q.weight'), None)
    if pe_key is not None:
        n_patches = state_dict[pe_key].shape[1]
        side = int(n_patches ** 0.5) * 16
        img_size = (side, side)
    else:
        img_size = fallback_img
    num_q = state_dict[q_key].shape[0] if q_key is not None else fallback_num_q
    return img_size, num_q


def load_eomt(ckpt_path, num_classes, lora=False,
              lora_r=8, lora_alpha=16, lora_target_modules=('qkv',), lora_dropout=0.05):
    '''Load an EoMT (standard or LoRA fine-tuned).
    For LoRA checkpoints the encoder is wrapped with PEFT BEFORE load_state_dict,
    using the same LoraConfig as Step 5 (r=8, alpha=16, target=`qkv`, dropout=0.05).
    '''
    raw = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    state_dict = raw['state_dict'] if isinstance(raw, dict) and 'state_dict' in raw else raw

    img_size, num_q = _detect_img_size_and_num_q(state_dict)
    print(f'  -> img_size={img_size}, num_q={num_q}, num_classes={num_classes}, lora={lora}')

    encoder = ViT(backbone_name='vit_base_patch14_reg4_dinov2', img_size=img_size)

    if lora:
        from peft import LoraConfig, get_peft_model
        lora_cfg = LoraConfig(
            r=lora_r, lora_alpha=lora_alpha,
            target_modules=list(lora_target_modules),
            lora_dropout=lora_dropout, bias='none',
        )
        encoder = get_peft_model(encoder, lora_cfg)

    network = EoMT(encoder=encoder, num_blocks=3, num_q=num_q, num_classes=num_classes)
    model = MaskClassificationSemantic(
        network=network, img_size=img_size, num_classes=num_classes,
        attn_mask_annealing_enabled=True,
        attn_mask_annealing_start_steps=[3317, 8292, 13268],
        attn_mask_annealing_end_steps=[6634, 11609, 16585],
        lr=1e-4, llrd=0.8, llrd_l2_enabled=True, lr_mult=1.0,
        weight_decay=0.05, poly_power=0.9, warmup_steps=[500, 1000],
        load_ckpt_class_head=bool(lora),
        ckpt_path=None,
    )
    # strict=False: the original .bin files are missing some Lightning module buffers.
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if lora:
        # Sanity check: for LoRA no backbone/LoRA keys should be missing
        lora_missing = [k for k in missing if 'lora_' in k or 'base_model.model' in k]
        if lora_missing:
            raise RuntimeError(
                f'Failing in LoRA upload, {len(lora_missing)} missing keys'
                f'(es: {lora_missing[:3]}). Check LoraConfig.'
            )
    model = model.to(DEVICE).eval()
    print(f'  Uploaded: {os.path.basename(ckpt_path)}')
    return model


In [ ]:
# Inference: returns EoMT's per-pixel logits (raw output).
#
# Internally to_per_pixel_logits_semantic computes:
#     L(x) = sum_n P_n(x) * M_n(x)
# where P_n is the query's class probs (post-softmax), M_n its mask (post-sigmoid),
# and K = the model's native num_classes (19 for CS/LoRA, 133 for COCO).
# So L is not a raw pre-softmax logit but a composite, roughly in [0, num_queries].
# We treat it as "logit-like" for the post-hoc methods (RbA paper, Eq. 2).
@torch.no_grad()
def get_pixel_logits(model, img):
    '''Forward pass with AMP, sliding-window for big images, then revert to the
    original resolution.
    Input:  img tensor CHW uint8 (the /255 normalization happens inside the model).
    Output: L tensor [K, H, W] float32 on DEVICE, K = model's native num_classes.
    '''
    with autocast(dtype=torch.float16, device_type=DEVICE.type):
        imgs      = [img.to(DEVICE)]
        img_sizes = [imgs[0].shape[-2:]]
        crops, origins      = model.window_imgs_semantic(imgs)
        ml_per_l, cl_per_l  = model(crops)
        mask_logits = F.interpolate(ml_per_l[-1], model.img_size, mode='bilinear')
        crop_logits = model.to_per_pixel_logits_semantic(mask_logits, cl_per_l[-1])
        logits      = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
    return logits[0].float()  # [K, H, W] on GPU


In [ ]:
# COCO 133-class -> Cityscapes 19-class remapping
#
# Why: the anomaly benchmarks here (SMIYC, FishyScapes, RoadAnomaly) define an
# anomaly as "a pixel not belonging to any of the 19 Cityscapes classes". The
# EoMT-COCO model was trained on 133 COCO panoptic classes and recognizes many
# typical road anomalies (cow, horse, elephant, bear, dog, ...) as known, so it
# can't flag them unless we align its class space to the benchmarks'.
#
# Fix (same as Step 4): we marginalize the COCO probabilities onto the 19
# Cityscapes classes, summing the probs of COCO classes that have an equivalent.
# COCO classes without one (including the typical anomalies) are just dropped
# from the sum: their probability mass disappears from L_cs, giving low scores on
# all 19 classes and thus a high anomaly score for the post-hoc methods.
#
# COCO_TO_CS is the same dict as inference_clean.ipynb (Step 4), reused so the
# evaluation pipeline stays consistent between Step 4 and Step 8.

IGNORE_INDEX = 255

# Panoptic COCO IDs (0-indexed) -> Cityscapes train IDs.
# Only 27 COCO classes have a Cityscapes equivalent; the other 106 map to
# IGNORE_INDEX (their probability mass is dropped).
COCO_TO_CS = {
    # -- Things (direct matches) ----------------------------------------------
     0: 11,  # person        -> person
     1: 18,  # bicycle       -> bicycle
     2: 13,  # car           -> car
     3: 17,  # motorcycle    -> motorcycle
     5: 15,  # bus           -> bus
     6: 16,  # train         -> train
     7: 14,  # truck         -> truck
     9:  6,  # traffic light -> traffic light
    11:  7,  # stop sign     -> traffic sign
    # -- Stuff (categorical merges) -------------------------------------------
     58:  8, # potted plant  -> vegetation
     88:  8, # flower        -> vegetation
     90:  9, # gravel        -> terrain
     91:  2, # house         -> building
    100:  0, # road          -> road
    102:  9, # sand          -> terrain
    109:  3, # wall-brick    -> wall
    110:  3, # wall-stone    -> wall
    111:  3, # wall-tile     -> wall
    112:  3, # wall-wood     -> wall
    116:  8, # tree-merged   -> vegetation
    117:  4, # fence-merged  -> fence
    119: 10, # sky-other     -> sky
    123:  1, # pavement      -> sidewalk
    125:  8, # grass-merged  -> vegetation
    126:  9, # dirt-merged   -> terrain
    129:  2, # building-other-> building
    131:  3, # wall-other    -> wall
}
# All remaining COCO classes -> IGNORE
for _cid in set(range(133)) - set(COCO_TO_CS):
    COCO_TO_CS[_cid] = IGNORE_INDEX


def build_remap_matrix(coco_to_cs, n_coco=133, n_cs=19, ignore_index=IGNORE_INDEX, device=DEVICE):
    '''Build the marginalization matrix R in R^{n_cs x n_coco} so that
        L_cs[k, h, w] = sum_{j: COCO_TO_CS[j]=k} L_coco[j, h, w]
    Columns for COCO classes mapped to IGNORE add to no row - their probability
    is dropped, which is exactly what lets us detect road animals as anomalies.
    '''
    R = torch.zeros(n_cs, n_coco, device=device)
    for c_coco, c_cs in coco_to_cs.items():
        if c_cs != ignore_index:
            R[c_cs, c_coco] = 1.0
    return R


# Build the remapping matrix once and reuse it across all images.
REMAP_COCO_TO_CS = build_remap_matrix(COCO_TO_CS)
print(f'Remap matrix shape: {tuple(REMAP_COCO_TO_CS.shape)}  '
      f'(mapped: {int(REMAP_COCO_TO_CS.sum().item())}/133 COCO classes)')


@torch.no_grad()
def get_pixel_logits_for_scoring(model, img, n_cls_native):
    '''Wrapper that ALWAYS returns logits in the 19-class Cityscapes space.
    Branches on n_cls_native:
      - 19  (CS / LoRA): return L as-is, already in the target space
      - 133 (COCO):      marginalize L via REMAP_COCO_TO_CS
    Output is [19, H, W] either way, so the post-hoc methods and the pixel-wise
    sub-sampling behave the same for all models.
    '''
    L = get_pixel_logits(model, img)               # [K, H, W], K in {19, 133}
    if n_cls_native == 19:
        return L
    if n_cls_native == 133:
        # einsum: per pixel, sum the probs of COCO classes that map to the same
        # Cityscapes class. Output: [19, H, W].
        return torch.einsum('kc,chw->khw', REMAP_COCO_TO_CS, L)
    raise ValueError(f'Classes number not supported: {n_cls_native}')


In [ ]:
# Anomaly scoring functions (torch, on GPU)
#
# Convention: in every method 'high score = anomalous pixel'.
# Expected input: L tensor [K, H, W] already in 19-class Cityscapes space
#                 (the remap cell guarantees this for all 3 models).
#
# Notes:
# - L is not a raw logit; it's L = sum_n P_n * M_n (post-softmax * post-sigmoid).
# - But the mask-transformer literature (RbA paper, Eq. 2) treats L as "logits"
#   and applies softmax/sigmoid to it freely, so we do the same for consistency.
# - MSP and MaxLogit are distinct by design (Hendrycks paper): MSP uses softmax,
#   MaxLogit doesn't. Without softmax in MSP the two collapse to an additive
#   constant and give identical AuPRC/FPR95.


def msp_score(L):
    '''Maximum Softmax Probability (Hendrycks & Gimpel 2017).
    Score = 1 - max_c softmax(L)_c.
    Pixel confident about a class -> low score (in-distribution).
    Pixel with a flat distribution -> high score (anomaly).
    '''
    return 1.0 - L.softmax(dim=0).max(dim=0).values


def maxlogit_score(L):
    '''MaxLogit (Hendrycks et al. 2022).
    Score = -max_c L_c.
    Pixel strongly claimed by some class -> high max(L) -> low score.
    All votes low -> low max(L) -> high score.
    On mask transformers, RbA (Table 3) applies MaxLogit to the aggregated L like this.
    '''
    return -L.max(dim=0).values


def maxentropy_score(L):
    '''Max-Entropy: Shannon entropy of softmax(L), normalized by log(K).
    Score in [0, 1]: 0 = peaked distribution (in-distribution),
                     1 = uniform distribution (max uncertainty, anomaly).
    '''
    p = L.softmax(dim=0)
    H = -(p * (p + 1e-12).log()).sum(dim=0) / float(np.log(L.shape[0]))
    return H


def rba_score(L):
    '''Rejected by All (Nayal et al. 2023), Eq. 5.
    Score = - sum_c sigmoid(L_c).
    Idea: each query acts as a one-vs-all classifier; if no query claims the pixel
    (sigmoid(L_c) low for all c), it belongs to no known class -> anomaly.
    '''
    return -torch.sigmoid(L).sum(dim=0)


# Dict used by the eval loop: add new methods here.
ANOMALY_FNS = {
    'MSP':        msp_score,
    'MaxLogit':   maxlogit_score,
    'MaxEntropy': maxentropy_score,
    'RbA':        rba_score,
}


In [ ]:
# Datasets, GT remap, sanity-check
#
# Each dataset uses its own anomaly label convention:
# - SMIYC RA-21, SMIYC RO-21, FS L&F, FS Static: 0 = inlier, 1 = anomaly, 255 = ignore
# - RoadAnomaly: 0 = inlier, 2 = anomaly, 255 = ignore  --> we remap 2 -> 1
VALIDATION_ROOT = '../Validation_Dataset'

DATASETS = {
    'SMIYC RA-21':  ('RoadAnomaly21',     f'{VALIDATION_ROOT}/RoadAnomaly21/images/*.png'),
    'SMIYC RO-21':  ('RoadObsticle21',    f'{VALIDATION_ROOT}/RoadObsticle21/images/*.webp'),
    'FS L&F':       ('FS_LostFound_full', f'{VALIDATION_ROOT}/FS_LostFound_full/images/*.png'),
    'FS Static':    ('fs_static',         f'{VALIDATION_ROOT}/fs_static/images/*.jpg'),
    'Road Anomaly': ('RoadAnomaly',       f'{VALIDATION_ROOT}/RoadAnomaly/images/*.jpg'),
}


def remap_gt(gt, dataset_key):
    '''Normalize the GT to {0, 1, 255} where 1 = anomaly.'''
    g = gt.copy()
    if dataset_key == 'RoadAnomaly':
        g = np.where(g == 2, 1, g)
    return g


def get_gt_path(img_path):
    '''The GT for each image lives in <dataset>/labels_masks/<stem>.png.'''
    gt = img_path.replace('images', 'labels_masks')
    for ext in ('.webp', '.jpg', '.jpeg'):
        gt = gt.replace(ext, '.png')
    return gt


for name, (key, pat) in DATASETS.items():
    print(f'{name:12s}: {len(glob.glob(pat))} images')


In [ ]:
# Logit caching + dataset evaluation
#
# Caching:
# - One .npz per image in cache_logits_final/<model>/<dataset>/<stem>.npz
# - Kept separate from the previous notebook's cache to avoid collisions (here the
#   COCO logits are stored already remapped to 19 classes, not 133).
# - Cached logits are float16 to save space on Drive.
# - Once cached, the temperature-scaling loop later needs no re-inference.
#
# Sub-sampling:
# - Concatenating all pixels of a dataset can easily exceed 10^8 points, too many
#   for average_precision_score. We cap at MAX_PIXELS_PER_DATASET with a uniform
#   random pick (seeded RNG for reproducibility).
CACHE_ROOT = './cache_logits_final'
os.makedirs(CACHE_ROOT, exist_ok=True)

MAX_PIXELS_PER_DATASET = 8_000_000
RNG = np.random.default_rng(SEED)


def cache_path_for(model_name, dataset_key, img_path):
    '''Path of the .npz used to save/load one image's logits.'''
    base = os.path.splitext(os.path.basename(img_path))[0]
    d = os.path.join(CACHE_ROOT, model_name, dataset_key)
    os.makedirs(d, exist_ok=True)
    return os.path.join(d, f'{base}.npz')


@torch.no_grad()
def get_or_compute_logits_at_gt(model, model_name, dataset_key, img_path,
                                gt_shape, n_cls_native):
    '''Return the [19, H_gt, W_gt] logits for the image.
    Loads them from cache if present, otherwise computes them, interpolates to the
    GT shape and saves them. Note: for COCO the logits are already remapped to 19
    classes by get_pixel_logits_for_scoring.
    '''
    cp = cache_path_for(model_name, dataset_key, img_path)
    if os.path.exists(cp):
        return np.load(cp)['L']

    img = torch.from_numpy(np.array(Image.open(img_path).convert('RGB'))).permute(2, 0, 1)
    L = get_pixel_logits_for_scoring(model, img, n_cls_native)   # [19, H, W]
    L = F.interpolate(L[None], size=tuple(gt_shape), mode='bilinear')[0]
    L_np = L.half().cpu().numpy()
    np.savez_compressed(cp, L=L_np)
    return L_np


def evaluate_dataset(model, model_name, dataset_key, display_name, image_glob,
                     n_cls_native, methods=('MSP', 'MaxLogit', 'MaxEntropy', 'RbA')):
    '''Run all post-hoc methods on one dataset.
    Returns dict {method: (AuPRC%, FPR95%)}.
    '''
    all_scores = {k: [] for k in methods}
    all_labels = []

    paths = sorted(glob.glob(os.path.expanduser(image_glob)))
    print(f'[{display_name}] {len(paths)} images founded')
    if not paths:
        return None

    for i, p in enumerate(paths):
        gt = np.array(Image.open(get_gt_path(p)))
        gt = remap_gt(gt, dataset_key)
        # Skip images with no anomaly (label=1) - they wouldn't add to AuPRC.
        if 1 not in np.unique(gt):
            continue

        L_np = get_or_compute_logits_at_gt(model, model_name, dataset_key, p,
                                           gt.shape, n_cls_native)
        L = torch.from_numpy(L_np).to(DEVICE).float()

        # Valid-pixel mask (excludes IGNORE = 255)
        valid_mask = torch.from_numpy(gt != 255).to(DEVICE)
        for k in methods:
            s = ANOMALY_FNS[k](L)
            all_scores[k].append(s[valid_mask].float().cpu().numpy())
        all_labels.append(gt[gt != 255])
        del L, valid_mask

        if (i + 1) % 20 == 0:
            print(f'  {i+1}/{len(paths)}')

    if not all_labels:
        print(f'  Warning: A valid GT is missing for {display_name}')
        return None

    # Sub-sample if we go over the pixel budget
    val_lab = np.concatenate(all_labels)
    if val_lab.size > MAX_PIXELS_PER_DATASET:
        idx = RNG.choice(val_lab.size, MAX_PIXELS_PER_DATASET, replace=False)
        val_lab = val_lab[idx]
    else:
        idx = None

    results = {}
    for k in methods:
        val_out = np.concatenate(all_scores[k])
        if idx is not None:
            val_out = val_out[idx]
        auprc = average_precision_score(val_lab, val_out) * 100.0
        fpr   = fpr_at_95_tpr(val_out, val_lab) * 100.0
        results[k] = (auprc, fpr)
    return results


In [ ]:
# Setup: results container (idempotent) + helper to evaluate one model
# Running the three eval cells below one at a time re-evaluates a single model.
# all_results is reused if it already exists, so re-running one cell doesn't lose
# the models already evaluated.

if 'all_results' not in globals():
    all_results = {}


def eval_one_model(model_name, ckpt_path, n_cls, is_lora):
    if not os.path.exists(ckpt_path):
        print(f'Skip {model_name}: missing checkpoint')
        return

    print(f'\n########## {model_name} (n_cls={n_cls}, lora={is_lora}) ##########')
    if n_cls == 133:
        print('  COCO model: logits will be remapped to 19 Cityscapes classes\n'
              '  through REMAP_COCO_TO_CS before scoring.')
    model = load_eomt(ckpt_path, num_classes=n_cls, lora=is_lora)
    all_results[model_name] = {}

    for display_name, (key, pat) in DATASETS.items():
        print(f'\n=== {display_name} ===')
        res = evaluate_dataset(model, model_name, key, display_name, pat, n_cls)
        if res is None:
            continue
        all_results[model_name][display_name] = res
        for m, (a, f) in res.items():
            print(f'  {m:11s}  AuPRC={a:6.2f}   FPR95={f:6.2f}')

    del model
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()


In [ ]:
# Evaluate EoMT-Cityscapes (native 19-class)
eval_one_model(*CHECKPOINTS[0])  # ('EoMT-Cityscapes', CS_CKPT, 19, False)


In [ ]:
# Evaluate EoMT-COCO (native 133-class, remapped to 19 on the fly)
eval_one_model(*CHECKPOINTS[1])  # ('EoMT-COCO', COCO_CKPT, 133, False)


In [ ]:
# Evaluate EoMT-FineTuned-LoRA (native 19-class, LoRA encoder)
eval_one_model(*CHECKPOINTS[2])  # ('EoMT-FineTuned-LoRA', FT_CKPT, 19, True)


In [ ]:
# Temperature scaling on MSP (reuses ONLY the cached logits: no re-inference)
#
# Canonical T-scaling (Guo et al. 2017):
#     score(x, T) = 1 - max_c softmax(L(x) / T)_c
#
# T has to act inside the softmax: it changes how "soft" the distribution is, so
# the different T values actually give different AuPRC/FPR95.
TEMPERATURES = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]


def best_T_msp(model_name, dataset_key, image_glob):
    '''For each T in TEMPERATURES, compute AuPRC and FPR95 of MSP@T from the
    cached logits. Returns (best_T, best_AuPRC, FPR95@best_T).
    '''
    paths = sorted(glob.glob(image_glob))
    scores_per_T = {T: [] for T in TEMPERATURES}
    labels = []

    for p in paths:
        gt = np.array(Image.open(get_gt_path(p)))
        gt = remap_gt(gt, dataset_key)
        if 1 not in np.unique(gt):
            continue
        cp = cache_path_for(model_name, dataset_key, p)
        if not os.path.exists(cp):
            # skip frames with no cache (the eval cell wasn't run for this model/dataset)
            continue
        L_np = np.load(cp)['L']
        L = torch.from_numpy(L_np).to(DEVICE).float()
        valid = torch.from_numpy(gt != 255).to(DEVICE)
        for T in TEMPERATURES:
            # canonical T-scaling: softmax(L/T), then max, then 1 - max
            s = 1.0 - (L / T).softmax(dim=0).max(dim=0).values
            scores_per_T[T].append(s[valid].cpu().numpy())
        labels.append(gt[gt != 255])
        del L, valid

    if not labels:
        return None, None, None

    y = np.concatenate(labels)
    if y.size > MAX_PIXELS_PER_DATASET:
        idx = RNG.choice(y.size, MAX_PIXELS_PER_DATASET, replace=False); y = y[idx]
    else:
        idx = None

    best_T, best_auprc, best_fpr = None, -1.0, None
    per_T_metrics = {}
    for T in TEMPERATURES:
        s = np.concatenate(scores_per_T[T])
        if idx is not None:
            s = s[idx]
        a = average_precision_score(y, s) * 100.0
        f = fpr_at_95_tpr(s, y) * 100.0
        per_T_metrics[T] = (a, f)
        if a > best_auprc:
            best_T, best_auprc, best_fpr = T, a, f
    return best_T, best_auprc, best_fpr, per_T_metrics


best_T_results = {}
all_T_results = {}
for model_name, _, _, _ in CHECKPOINTS:
    best_T_results[model_name] = {}
    all_T_results[model_name]  = {}
    for display_name, (key, pat) in DATASETS.items():
        out = best_T_msp(model_name, key, pat)
        if out is None or out[0] is None:
            continue
        T, a, f, per_T = out
        best_T_results[model_name][display_name] = (T, a, f)
        all_T_results[model_name][display_name]  = per_T
        print(f'{model_name:22s} {display_name:14s}  best T={T:>4}  '
              f'AuPRC={a:6.2f}  FPR95={f:6.2f}')


In [ ]:
# Summary CSV table (models x methods x datasets, with MSP@bestT)
import pandas as pd

METHODS  = ['MSP', 'MaxLogit', 'MaxEntropy', 'RbA']
DS_ORDER = ['SMIYC RA-21', 'SMIYC RO-21', 'FS L&F', 'FS Static', 'Road Anomaly']

rows = []
for model_name in all_results:
    # One row per base method
    for m in METHODS:
        row = {'Model': model_name, 'Method': m}
        for ds in DS_ORDER:
            entry = all_results[model_name].get(ds, {}).get(m)
            row[f'{ds} AuPRC'] = round(entry[0], 2) if entry else None
            row[f'{ds} FPR95'] = round(entry[1], 2) if entry else None
        rows.append(row)

    # MSP@bestT row (canonical T-scaling)
    row = {'Model': model_name, 'Method': 'MSP@bestT'}
    for ds in DS_ORDER:
        entry = best_T_results.get(model_name, {}).get(ds)
        if entry:
            T_best, a, f = entry
            row[f'{ds} AuPRC'] = round(a, 2)
            row[f'{ds} FPR95'] = round(f, 2)
        else:
            row[f'{ds} AuPRC'] = None
            row[f'{ds} FPR95'] = None

        if entry:
            row[f'{ds} bestT'] = entry[0]
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv('step8_results_final.csv', index=False)
print('Saved in: step8_results_final.csv\n')
df


In [ ]:

# This cell builds it from all_T_results, one CSV per model.

for model_name in all_results:
    if model_name not in all_T_results:
        continue
    rows = []
    # Default MSP row (T=1.0, equivalent to the base MSP method).
    base = {'Method': 'MSP'}
    for ds in DS_ORDER:
        entry = all_results[model_name].get(ds, {}).get('MSP')
        base[f'{ds} AuPRC'] = round(entry[0], 2) if entry else None
        base[f'{ds} FPR95'] = round(entry[1], 2) if entry else None
    rows.append(base)

    # One row per T
    for T in TEMPERATURES:
        r = {'Method': f'MSP(t={T})'}
        for ds in DS_ORDER:
            per_T = all_T_results[model_name].get(ds, {})
            entry = per_T.get(T)
            r[f'{ds} AuPRC'] = round(entry[0], 2) if entry else None
            r[f'{ds} FPR95'] = round(entry[1], 2) if entry else None
        rows.append(r)

    # MSP(best t) row
    rbest = {'Method': 'MSP(best t)'}
    for ds in DS_ORDER:
        entry = best_T_results.get(model_name, {}).get(ds)
        if entry:
            T_best, a, f = entry
            rbest[f'{ds} AuPRC'] = round(a, 2)
            rbest[f'{ds} FPR95'] = round(f, 2)
            rbest[f'{ds} bestT'] = T_best
        else:
            rbest[f'{ds} AuPRC'] = None
            rbest[f'{ds} FPR95'] = None
    rows.append(rbest)

    df_T = pd.DataFrame(rows)
    fn = f'step8_tscaling_{model_name}.csv'
    df_T.to_csv(fn, index=False)
    print(f'Saved: {fn}')
    display(df_T) if 'display' in dir() else print(df_T)
